# v3g YOLO Inference-Only Proposal Analysis

This notebook does not train a model.

It evaluates the trained YOLOv8n `v3g_li_manual_keep_yolov8n_640` model as a proposal generator for:

```text
LiDAR tile -> YOLO candidate bbox -> segmentation refinement
```

Outputs:

- `threshold_sweep.csv`
- `proposal_coverage.csv`
- `false_negative_analysis.csv`
- `summary.md`


## 1. Configuration


In [ ]:
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
WORK_DATASETS_DIR = WORK_ROOT / "datasets"
OUTPUT_ROOT = WORK_ROOT / "v3g_inference_analysis"
ANALYSIS_DIR = OUTPUT_ROOT / "analysis"
SCRIPT_DIR = OUTPUT_ROOT / "scripts"
WEIGHTS_WORK_DIR = OUTPUT_ROOT / "weights"

DATASET_FOLDER = "dataset_yolo_bbox_v3g_li_medium_manual_keep_only"
PREBUILT_DATASET_DIR = Path(
    "/kaggle/input/datasets/matanerdy/detection-dataset/dataset_yolo_bbox_v3g_li_medium_manual_keep_only"
)
DATASET_WORK_DIR = WORK_DATASETS_DIR / DATASET_FOLDER
METADATA_PATH = DATASET_WORK_DIR / "metadata.csv"

# Update this if the trained v3g weights are uploaded under a different Kaggle input path.
PREFERRED_WEIGHTS_PATHS = [
    Path("/kaggle/input/datasets/matanerdy/detection-dataset/v3g_li_manual_keep_yolov8n_640_best.pt"),
    Path("/kaggle/input/datasets/matanerdy/detection-dataset/best.pt"),
]

IMGSZ = 640
NMS_IOU = 0.50
DEVICE = 0

WORK_DATASETS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_WORK_DIR.mkdir(parents=True, exist_ok=True)


## 2. Install Dependencies


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pandas", "pillow", "matplotlib"],
    check=True,
)


## 3. Copy Dataset to Working Directory


In [ ]:
import shutil
import zipfile

import pandas as pd

def find_dataset_dir(folder_name: str) -> Path | None:
    if PREBUILT_DATASET_DIR.exists():
        return PREBUILT_DATASET_DIR
    for metadata in KAGGLE_INPUT_ROOT.rglob("metadata.csv"):
        parent = metadata.parent
        if parent.name == folder_name and (parent / "images").exists() and (parent / "labels").exists():
            return parent
    return None

def find_dataset_zip(folder_name: str) -> Path | None:
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob(f"{folder_name}.zip"))
    return candidates[0] if candidates else None

source_dataset = find_dataset_dir(DATASET_FOLDER)
if DATASET_WORK_DIR.exists():
    shutil.rmtree(DATASET_WORK_DIR)

if source_dataset is not None:
    print("Copying dataset:", source_dataset)
    shutil.copytree(source_dataset, DATASET_WORK_DIR)
else:
    zip_path = find_dataset_zip(DATASET_FOLDER)
    if zip_path is None:
        raise FileNotFoundError(
            f"Attach Kaggle input containing {DATASET_FOLDER}/ or {DATASET_FOLDER}.zip"
        )
    print("Extracting dataset:", zip_path)
    unzip_root = WORK_DATASETS_DIR / "_unzipped_v3g"
    if unzip_root.exists():
        shutil.rmtree(unzip_root)
    unzip_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(unzip_root)
    candidates = [p.parent for p in unzip_root.rglob("metadata.csv") if p.parent.name == DATASET_FOLDER]
    extracted = candidates[0] if candidates else next(unzip_root.rglob("metadata.csv")).parent
    shutil.copytree(extracted, DATASET_WORK_DIR)

meta = pd.read_csv(METADATA_PATH)
images = meta.drop_duplicates("image").copy()
boxes = meta[meta["class_name"].notna()].copy()
print("Dataset:", DATASET_WORK_DIR)
print("Images:", len(images))
print("Positive:", int(images["is_positive"].astype(bool).sum()))
print("Negative:", int((~images["is_positive"].astype(bool)).sum()))
print("BBox:", len(boxes))
print("\nVal split:")
print(images[images["split"].eq("val")].groupby("is_positive").size())


## 4. Locate Weights


In [ ]:
import shutil

def find_weights() -> Path:
    for path in PREFERRED_WEIGHTS_PATHS:
        if path.exists():
            return path
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob("*.pt"))
    if not candidates:
        raise FileNotFoundError(
            "Attach the trained v3g YOLO best.pt as a Kaggle input. "
            "If needed, update PREFERRED_WEIGHTS_PATHS in the config cell."
        )
    preferred = [
        p for p in candidates
        if "v3g" in str(p).lower()
        or "manual" in str(p).lower()
        or "keep" in str(p).lower()
        or "kurgan" in str(p).lower()
    ]
    return preferred[0] if preferred else candidates[0]

source_weights = find_weights()
weights_path = WEIGHTS_WORK_DIR / "v3g_li_manual_keep_yolov8n_640_best.pt"
shutil.copy2(source_weights, weights_path)
print("Source weights:", source_weights)
print("Working weights:", weights_path)
print("Size MB:", round(weights_path.stat().st_size / (1024 * 1024), 2))


## 5. Write Analysis Script


In [ ]:
SWEEP_SCRIPT = 'from __future__ import annotations\n\nimport argparse\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom ultralytics import YOLO\n\nos.environ.setdefault("MPLBACKEND", "Agg")\nos.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")\n\ntry:\n    import matplotlib\n\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\nexcept Exception:  # pragma: no cover\n    plt = None\n\n\nCONF_SWEEP = [0.50, 0.25, 0.10, 0.05, 0.03, 0.01, 0.005, 0.003, 0.001]\nMATCH_IOU = 0.50\nCOVERAGE_IOU = 0.30\nPROPOSAL_IOUS = [0.10, 0.20, 0.30, 0.50]\nRECOMMENDATION_CONFS = [0.05, 0.03, 0.01, 0.005]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Inference-only threshold/proposal analysis for v3g YOLO baseline.")\n    parser.add_argument("--metadata", type=Path, required=True)\n    parser.add_argument("--weights", type=Path, required=True)\n    parser.add_argument("--out-dir", type=Path, required=True)\n    parser.add_argument("--imgsz", type=int, default=640)\n    parser.add_argument("--nms-iou", type=float, default=0.50)\n    parser.add_argument("--device", default=None)\n    return parser.parse_args()\n\n\ndef resolve_image_path(row: pd.Series, dataset_dir: Path) -> Path:\n    split = str(row["split"])\n    image_name = str(row.get("image_name") or Path(str(row["image"])).name)\n    candidates = [\n        dataset_dir / "images" / split / image_name,\n        Path(str(row["image"])),\n        dataset_dir / "images" / split / Path(str(row["image"])).name,\n    ]\n    for candidate in candidates:\n        if candidate.exists():\n            return candidate.resolve()\n    return candidates[0].resolve()\n\n\ndef load_validation_gt(metadata_path: Path) -> tuple[pd.DataFrame, dict[str, pd.DataFrame], list[Path]]:\n    metadata_path = metadata_path.resolve()\n    dataset_dir = metadata_path.parent\n    meta = pd.read_csv(metadata_path)\n    val = meta[meta["split"].astype(str).str.lower().eq("val")].copy()\n    if val.empty:\n        raise ValueError("No validation rows in metadata.csv")\n\n    val["image_path"] = val.apply(lambda row: resolve_image_path(row, dataset_dir), axis=1)\n    missing = sorted({str(path) for path in val["image_path"] if not path.exists()})\n    if missing:\n        preview = "\\n".join(missing[:10])\n        raise FileNotFoundError(f"Validation images are missing. First paths:\\n{preview}")\n    val["image_key"] = val["image_path"].map(lambda p: str(p.resolve()))\n    val_images = val.drop_duplicates("image_key").copy()\n\n    gt = val[val["class_name"].notna()].copy()\n    gt["gt_id"] = np.arange(len(gt))\n    gt["bbox_width_px"] = pd.to_numeric(gt["bbox_width_px"], errors="coerce") if "bbox_width_px" in gt else np.nan\n    gt["bbox_height_px"] = pd.to_numeric(gt["bbox_height_px"], errors="coerce") if "bbox_height_px" in gt else np.nan\n    if gt["bbox_width_px"].isna().all():\n        gt["bbox_width_px"] = pd.to_numeric(gt["bbox_x2_px"], errors="coerce") - pd.to_numeric(gt["bbox_x1_px"], errors="coerce")\n    if gt["bbox_height_px"].isna().all():\n        gt["bbox_height_px"] = pd.to_numeric(gt["bbox_y2_px"], errors="coerce") - pd.to_numeric(gt["bbox_y1_px"], errors="coerce")\n\n    xyxy = []\n    for _, row in gt.iterrows():\n        image_path = Path(row["image_key"])\n        width, height = Image.open(image_path).size\n        if {"yolo_xc", "yolo_yc", "yolo_w", "yolo_h"}.issubset(gt.columns) and pd.notna(row.get("yolo_xc")):\n            xc = float(row["yolo_xc"]) * width\n            yc = float(row["yolo_yc"]) * height\n            bw = float(row["yolo_w"]) * width\n            bh = float(row["yolo_h"]) * height\n            xyxy.append((xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2))\n        else:\n            xyxy.append((float(row["bbox_x1_px"]), float(row["bbox_y1_px"]), float(row["bbox_x2_px"]), float(row["bbox_y2_px"])))\n    gt["bbox_xyxy"] = xyxy\n\n    gt_by_image = {key: group.sort_values("gt_id").reset_index(drop=True) for key, group in gt.groupby("image_key")}\n    image_paths = [Path(p).resolve() for p in val_images["image_path"]]\n    return val, gt_by_image, image_paths\n\n\ndef box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    if len(a) == 0 or len(b) == 0:\n        return np.zeros((len(a), len(b)), dtype=float)\n    x1 = np.maximum(a[:, None, 0], b[None, :, 0])\n    y1 = np.maximum(a[:, None, 1], b[None, :, 1])\n    x2 = np.minimum(a[:, None, 2], b[None, :, 2])\n    y2 = np.minimum(a[:, None, 3], b[None, :, 3])\n    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)\n    area_a = np.maximum(0, a[:, 2] - a[:, 0]) * np.maximum(0, a[:, 3] - a[:, 1])\n    area_b = np.maximum(0, b[:, 2] - b[:, 0]) * np.maximum(0, b[:, 3] - b[:, 1])\n    union = area_a[:, None] + area_b[None, :] - inter\n    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)\n\n\ndef predict(model: YOLO, image_paths: list[Path], conf: float, nms_iou: float, imgsz: int, device: str | None) -> pd.DataFrame:\n    kwargs = {\n        "source": [str(p) for p in image_paths],\n        "imgsz": imgsz,\n        "conf": conf,\n        "iou": nms_iou,\n        "verbose": False,\n        "save": False,\n        "stream": False,\n    }\n    if device:\n        kwargs["device"] = device\n    results = model.predict(**kwargs)\n    rows = []\n    for result_idx, result in enumerate(results):\n        image_key = str(image_paths[result_idx].resolve())\n        if result.boxes is None or len(result.boxes) == 0:\n            continue\n        xyxy = result.boxes.xyxy.cpu().numpy()\n        confs = result.boxes.conf.cpu().numpy()\n        for pred_idx, (box, score) in enumerate(zip(xyxy, confs)):\n            rows.append(\n                {\n                    "image_key": image_key,\n                    "pred_id": f"{Path(image_key).name}:{pred_idx}",\n                    "x1": float(box[0]),\n                    "y1": float(box[1]),\n                    "x2": float(box[2]),\n                    "y2": float(box[3]),\n                    "confidence": float(score),\n                    "bbox_width_px": float(box[2] - box[0]),\n                    "bbox_height_px": float(box[3] - box[1]),\n                    "bbox_area_px": float((box[2] - box[0]) * (box[3] - box[1])),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef object_fields(row: pd.Series | dict) -> dict:\n    get = row.get\n    return {\n        "gt_id": get("gt_id"),\n        "image_id": get("image_id") or Path(str(get("image_key"))).stem,\n        "image_key": get("image_key"),\n        "source_class_name": get("source_class_name"),\n        "bbox_area_px": get("bbox_area_px"),\n        "bbox_width_px": get("bbox_width_px"),\n        "bbox_height_px": get("bbox_height_px"),\n        "bbox_x1_px": get("bbox_x1_px"),\n        "bbox_y1_px": get("bbox_y1_px"),\n        "bbox_x2_px": get("bbox_x2_px"),\n        "bbox_y2_px": get("bbox_y2_px"),\n        "region": get("region"),\n        "modality": get("modality"),\n    }\n\n\ndef evaluate(\n    predictions: pd.DataFrame,\n    gt_by_image: dict[str, pd.DataFrame],\n    image_paths: list[Path],\n    conf: float,\n    nms_iou: float,\n) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    tp_rows = []\n    fp_rows = []\n    fn_rows = []\n    covered_gt = 0\n    total_gt = sum(len(group) for group in gt_by_image.values())\n    pred_by_image = {key: group.sort_values("confidence", ascending=False).reset_index(drop=True) for key, group in predictions.groupby("image_key")} if not predictions.empty else {}\n\n    for image_path in image_paths:\n        image_key = str(image_path.resolve())\n        gt_group = gt_by_image.get(image_key, pd.DataFrame()).copy()\n        pred_group = pred_by_image.get(image_key, pd.DataFrame()).copy()\n        gt_boxes = np.array(gt_group["bbox_xyxy"].tolist(), dtype=float) if not gt_group.empty else np.empty((0, 4))\n        pred_boxes = pred_group[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float) if not pred_group.empty else np.empty((0, 4))\n        ious = box_iou(pred_boxes, gt_boxes)\n\n        if len(pred_boxes) and len(gt_boxes):\n            covered_gt += int((ious.max(axis=0) >= COVERAGE_IOU).sum())\n\n        candidates = []\n        if len(pred_boxes) and len(gt_boxes):\n            pred_order = pred_group["confidence"].to_numpy()\n            for pred_idx in range(len(pred_boxes)):\n                for gt_idx in range(len(gt_boxes)):\n                    if ious[pred_idx, gt_idx] >= MATCH_IOU:\n                        candidates.append((float(ious[pred_idx, gt_idx]), float(pred_order[pred_idx]), pred_idx, gt_idx))\n        matched_pred = set()\n        matched_gt = set()\n        for iou_value, score, pred_idx, gt_idx in sorted(candidates, reverse=True):\n            if pred_idx in matched_pred or gt_idx in matched_gt:\n                continue\n            matched_pred.add(pred_idx)\n            matched_gt.add(gt_idx)\n            gt_row = gt_group.iloc[gt_idx]\n            pred_row = pred_group.iloc[pred_idx]\n            tp_rows.append(\n                {\n                    **object_fields(gt_row),\n                    "conf": conf,\n                    "match_iou": iou_value,\n                    "prediction_confidence": float(pred_row["confidence"]),\n                    "pred_bbox_area_px": float(pred_row["bbox_area_px"]),\n                }\n            )\n\n        for pred_idx, pred_row in pred_group.iterrows():\n            if pred_idx in matched_pred:\n                continue\n            best_gt_iou = float(ious[pred_idx].max()) if len(gt_boxes) else 0.0\n            fp_rows.append(\n                {\n                    "conf": conf,\n                    "image_key": image_key,\n                    "prediction_confidence": float(pred_row["confidence"]),\n                    "best_gt_iou": best_gt_iou,\n                    "bbox_area_px": float(pred_row["bbox_area_px"]),\n                    "bbox_width_px": float(pred_row["bbox_width_px"]),\n                    "bbox_height_px": float(pred_row["bbox_height_px"]),\n                }\n            )\n\n        for gt_idx, gt_row in gt_group.iterrows():\n            if gt_idx in matched_gt:\n                continue\n            best_pred_iou = float(ious[:, gt_idx].max()) if len(pred_boxes) else 0.0\n            fn_rows.append({**object_fields(gt_row), "conf": conf, "best_pred_iou_at_conf": best_pred_iou})\n\n    tp = len(tp_rows)\n    fp = len(fp_rows)\n    fn = len(fn_rows)\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    metrics = {\n        "conf": conf,\n        "nms_iou": nms_iou,\n        "TP": tp,\n        "FP": fp,\n        "FN": fn,\n        "Precision": precision,\n        "Recall": recall,\n        "F1": f1,\n        "covered_gt": covered_gt,\n        "total_gt": total_gt,\n        "coverage_rate": covered_gt / total_gt if total_gt else 0.0,\n        "FP_per_image": fp / len(image_paths) if image_paths else 0.0,\n    }\n    return metrics, pd.DataFrame(tp_rows), pd.DataFrame(fp_rows), pd.DataFrame(fn_rows)\n\n\ndef max_iou_per_gt(predictions: pd.DataFrame, gt_by_image: dict[str, pd.DataFrame], image_paths: list[Path]) -> pd.DataFrame:\n    rows = []\n    pred_by_image = {key: group.reset_index(drop=True) for key, group in predictions.groupby("image_key")} if not predictions.empty else {}\n    for image_path in image_paths:\n        image_key = str(image_path.resolve())\n        gt_group = gt_by_image.get(image_key, pd.DataFrame()).copy()\n        pred_group = pred_by_image.get(image_key, pd.DataFrame()).copy()\n        gt_boxes = np.array(gt_group["bbox_xyxy"].tolist(), dtype=float) if not gt_group.empty else np.empty((0, 4))\n        pred_boxes = pred_group[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float) if not pred_group.empty else np.empty((0, 4))\n        ious = box_iou(pred_boxes, gt_boxes)\n        for gt_idx, gt_row in gt_group.iterrows():\n            max_iou = float(ious[:, gt_idx].max()) if len(pred_boxes) else 0.0\n            best_conf = float(pred_group.iloc[int(np.argmax(ious[:, gt_idx]))]["confidence"]) if len(pred_boxes) else np.nan\n            rows.append({**object_fields(gt_row), "max_iou_any_prediction": max_iou, "best_prediction_confidence": best_conf})\n    return pd.DataFrame(rows)\n\n\ndef describe_objects(df: pd.DataFrame, group_name: str) -> pd.DataFrame:\n    rows = []\n    for metric in ["bbox_area_px", "bbox_width_px", "bbox_height_px"]:\n        values = pd.to_numeric(df[metric], errors="coerce").dropna() if metric in df else pd.Series(dtype=float)\n        if values.empty:\n            continue\n        rows.append(\n            {\n                "group": group_name,\n                "metric": metric,\n                "count": int(len(values)),\n                "mean": float(values.mean()),\n                "median": float(values.median()),\n                "p10": float(values.quantile(0.10)),\n                "p25": float(values.quantile(0.25)),\n                "p75": float(values.quantile(0.75)),\n                "p90": float(values.quantile(0.90)),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef write_plots(threshold_df: pd.DataFrame, proposal_gt: pd.DataFrame, found: pd.DataFrame, missed: pd.DataFrame, out_dir: Path) -> None:\n    if plt is None:\n        return\n    fig, ax = plt.subplots(figsize=(8, 5))\n    for col in ["Precision", "Recall", "F1", "coverage_rate"]:\n        ax.plot(threshold_df["conf"], threshold_df[col], marker="o", label=col)\n    ax.set_xscale("log")\n    ax.invert_xaxis()\n    ax.grid(True, alpha=0.25)\n    ax.set_xlabel("confidence threshold")\n    ax.set_ylabel("score")\n    ax.set_title("v3g threshold sweep")\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out_dir / "threshold_sweep.png", dpi=180)\n    plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(8, 5))\n    ax.hist(proposal_gt["max_iou_any_prediction"], bins=24, alpha=0.85)\n    ax.set_xlabel("max IoU per GT")\n    ax.set_ylabel("GT objects")\n    ax.set_title("Proposal max IoU distribution")\n    ax.grid(True, alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_dir / "max_iou_per_gt_distribution.png", dpi=180)\n    plt.close(fig)\n\n    combined = []\n    if not found.empty:\n        tmp = found.copy()\n        tmp["group"] = "found"\n        combined.append(tmp)\n    if not missed.empty:\n        tmp = missed.copy()\n        tmp["group"] = "missed"\n        combined.append(tmp)\n    if combined:\n        both = pd.concat(combined, ignore_index=True)\n        for metric in ["bbox_area_px", "bbox_width_px", "bbox_height_px"]:\n            fig, ax = plt.subplots(figsize=(8, 5))\n            for group, group_df in both.groupby("group"):\n                values = pd.to_numeric(group_df[metric], errors="coerce").dropna()\n                if not values.empty:\n                    ax.hist(values, bins=24, alpha=0.55, label=group)\n            ax.set_title(f"{metric}: found vs missed")\n            ax.set_xlabel(metric)\n            ax.set_ylabel("GT objects")\n            ax.grid(True, alpha=0.25)\n            ax.legend()\n            fig.tight_layout()\n            fig.savefig(out_dir / f"{metric}_found_vs_missed.png", dpi=180)\n            plt.close(fig)\n\n\ndef markdown_table(df: pd.DataFrame, floatfmt: str = ".4f") -> str:\n    if df.empty:\n        return "_No rows._"\n    formatted = df.copy()\n    for col in formatted.columns:\n        if pd.api.types.is_float_dtype(formatted[col]):\n            formatted[col] = formatted[col].map(lambda x: format(x, floatfmt))\n    formatted = formatted.fillna("")\n    cols = [str(c) for c in formatted.columns]\n    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]\n    for _, row in formatted.iterrows():\n        lines.append("| " + " | ".join(str(row[c]) for c in formatted.columns) + " |")\n    return "\\n".join(lines)\n\n\ndef write_summary(\n    out_path: Path,\n    threshold_df: pd.DataFrame,\n    proposal_cov: pd.DataFrame,\n    size_stats: pd.DataFrame,\n    rec_df: pd.DataFrame,\n    selected_conf: float,\n) -> None:\n    selected = threshold_df[threshold_df["conf"].eq(selected_conf)].iloc[0]\n    high = threshold_df[threshold_df["conf"].eq(0.25)].iloc[0]\n    low = threshold_df[threshold_df["conf"].eq(0.001)].iloc[0]\n\n    if float(low["coverage_rate"]) > float(high["coverage_rate"]) + 0.10:\n        verdict = "B. Модель умеет предлагать часть кандидатов, но стандартный threshold слишком высокий."\n    else:\n        verdict = "A. Основное ограничение похоже не только на threshold: модель часто не генерирует достаточно близкие кандидаты."\n\n    lines = [\n        "# v3g Inference-Only Proposal Analysis",\n        "",\n        "## Threshold Sweep",\n        "",\n        markdown_table(threshold_df),\n        "",\n        "## Proposal Coverage",\n        "",\n        markdown_table(proposal_cov),\n        "",\n        "## Proposal Mode Candidates",\n        "",\n        markdown_table(rec_df),\n        "",\n        "## Found vs Missed Object Size",\n        "",\n        markdown_table(size_stats, floatfmt=".2f"),\n        "",\n        "## Итоговый вывод",\n        "",\n        verdict,\n        "",\n        f"При стандартном `conf=0.25` модель дает Recall `{high[\'Recall\']:.3f}` и coverage `{high[\'coverage_rate\']:.3f}`. "\n        f"При экстремально низком `conf=0.001` Recall становится `{low[\'Recall\']:.3f}`, coverage `{low[\'coverage_rate\']:.3f}`, "\n        f"но FP/image растет до `{low[\'FP_per_image\']:.2f}`.",\n        "",\n        f"Для proposal режима выбран рабочий кандидат `conf={selected_conf}`: "\n        f"Recall `{selected[\'Recall\']:.3f}`, coverage `{selected[\'coverage_rate\']:.3f}`, FP/image `{selected[\'FP_per_image\']:.2f}`. "\n        "Это значение стоит сравнивать визуально с соседними `0.03`, `0.01`, `0.005`, потому что итоговый выбор зависит от того, сколько ложных crop-кандидатов выдержит следующий segmentation/refinement этап.",\n        "",\n        "Если coverage на низких threshold заметно выше обычного Recall, то связка `LiDAR -> YOLO proposal generation -> segmentation refinement` выглядит более реалистичной, чем попытка сразу получить высокий mAP на детекторе. "\n        "Если же coverage почти не растет, bottleneck остается в данных/разметке/визуальной различимости объектов, а не только в threshold.",\n        "",\n        "## Saved Files",\n        "",\n        "- `threshold_sweep.csv`",\n        "- `proposal_coverage.csv`",\n        "- `false_negative_analysis.csv`",\n        "- `found_object_analysis.csv`",\n        "- `object_size_found_vs_missed.csv`",\n        "- `max_iou_per_gt.csv`",\n        "- `summary.md`",\n        "",\n    ]\n    out_path.write_text("\\n".join(lines), encoding="utf-8")\n\n\ndef main() -> None:\n    args = parse_args()\n    args.out_dir.mkdir(parents=True, exist_ok=True)\n\n    val, gt_by_image, image_paths = load_validation_gt(args.metadata)\n    model = YOLO(str(args.weights))\n\n    all_metrics = []\n    tp_by_conf: dict[float, pd.DataFrame] = {}\n    fn_by_conf: dict[float, pd.DataFrame] = {}\n    fp_by_conf: dict[float, pd.DataFrame] = {}\n    pred_by_conf: dict[float, pd.DataFrame] = {}\n    for conf in CONF_SWEEP:\n        print(f"Predict/evaluate conf={conf}")\n        predictions = predict(model, image_paths, conf=conf, nms_iou=args.nms_iou, imgsz=args.imgsz, device=args.device)\n        metrics, tp, fp, fn = evaluate(predictions, gt_by_image, image_paths, conf=conf, nms_iou=args.nms_iou)\n        all_metrics.append(metrics)\n        tp_by_conf[conf] = tp\n        fp_by_conf[conf] = fp\n        fn_by_conf[conf] = fn\n        pred_by_conf[conf] = predictions\n\n    threshold_df = pd.DataFrame(all_metrics)\n    threshold_df.to_csv(args.out_dir / "threshold_sweep.csv", index=False)\n\n    low_conf = min(CONF_SWEEP)\n    proposal_gt = max_iou_per_gt(pred_by_conf[low_conf], gt_by_image, image_paths)\n    proposal_gt.to_csv(args.out_dir / "max_iou_per_gt.csv", index=False)\n    proposal_rows = []\n    total_gt = len(proposal_gt)\n    for iou_threshold in PROPOSAL_IOUS:\n        covered = int((proposal_gt["max_iou_any_prediction"] >= iou_threshold).sum())\n        proposal_rows.append(\n            {\n                "prediction_conf": low_conf,\n                "nms_iou": args.nms_iou,\n                "coverage_iou_threshold": iou_threshold,\n                "covered_gt": covered,\n                "total_gt": total_gt,\n                "coverage_rate": covered / total_gt if total_gt else 0.0,\n            }\n        )\n    proposal_cov = pd.DataFrame(proposal_rows)\n    proposal_cov.to_csv(args.out_dir / "proposal_coverage.csv", index=False)\n\n    rec_df = threshold_df[threshold_df["conf"].isin(RECOMMENDATION_CONFS)].copy()\n    rec_df = rec_df[["conf", "TP", "FP", "FN", "Precision", "Recall", "F1", "coverage_rate", "FP_per_image"]]\n    rec_df.to_csv(args.out_dir / "proposal_mode_candidates.csv", index=False)\n\n    # Use the best F1 among the proposal-relevant thresholds as the main FN table.\n    selected_row = rec_df.sort_values(["F1", "coverage_rate"], ascending=False).iloc[0]\n    selected_conf = float(selected_row["conf"])\n    false_negative = fn_by_conf[selected_conf].copy()\n    found = tp_by_conf[selected_conf].copy()\n    false_negative[["image_id", "bbox_area_px", "bbox_width_px", "bbox_height_px", "source_class_name"]].to_csv(\n        args.out_dir / "false_negative_analysis.csv", index=False\n    )\n    found.to_csv(args.out_dir / "found_object_analysis.csv", index=False)\n    fp_by_conf[selected_conf].to_csv(args.out_dir / "false_positive_analysis.csv", index=False)\n\n    all_fn = pd.concat([df for df in fn_by_conf.values() if not df.empty], ignore_index=True) if any(not df.empty for df in fn_by_conf.values()) else pd.DataFrame()\n    all_fn.to_csv(args.out_dir / "false_negative_analysis_all_thresholds.csv", index=False)\n\n    size_stats = pd.concat(\n        [describe_objects(found, "found"), describe_objects(false_negative, "missed")],\n        ignore_index=True,\n    )\n    size_stats.to_csv(args.out_dir / "object_size_found_vs_missed.csv", index=False)\n\n    write_plots(threshold_df, proposal_gt, found, false_negative, args.out_dir)\n    write_summary(args.out_dir / "summary.md", threshold_df, proposal_cov, size_stats, rec_df, selected_conf)\n    print("Saved analysis to:", args.out_dir)\n\n\nif __name__ == "__main__":\n    main()\n'

sweep_script_path = SCRIPT_DIR / "sweep_v3g_inference.py"
sweep_script_path.write_text(SWEEP_SCRIPT, encoding="utf-8")
print("Sweep script:", sweep_script_path)


## 6. Run Inference-Only Analysis


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(SCRIPT_DIR / "sweep_v3g_inference.py"),
    "--metadata",
    str(METADATA_PATH),
    "--weights",
    str(weights_path),
    "--out-dir",
    str(ANALYSIS_DIR),
    "--imgsz",
    str(IMGSZ),
    "--nms-iou",
    str(NMS_IOU),
    "--device",
    str(DEVICE),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 7. Inspect Tables


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

threshold_sweep = pd.read_csv(ANALYSIS_DIR / "threshold_sweep.csv")
proposal_coverage = pd.read_csv(ANALYSIS_DIR / "proposal_coverage.csv")
fn_analysis = pd.read_csv(ANALYSIS_DIR / "false_negative_analysis.csv")
size_stats = pd.read_csv(ANALYSIS_DIR / "object_size_found_vs_missed.csv")

display(Markdown("### Threshold Sweep"))
display(threshold_sweep)
display(Markdown("### Proposal Coverage"))
display(proposal_coverage)
display(Markdown("### Found vs Missed Object Size"))
display(size_stats)
display(Markdown(f"False negatives saved: `{len(fn_analysis)}` rows"))


## 8. Show Plots


In [ ]:
from IPython.display import Image, display

for name in [
    "threshold_sweep.png",
    "max_iou_per_gt_distribution.png",
    "bbox_area_px_found_vs_missed.png",
    "bbox_width_px_found_vs_missed.png",
    "bbox_height_px_found_vs_missed.png",
]:
    path = ANALYSIS_DIR / name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))
    else:
        print("Missing:", path)


## 9. Summary


In [ ]:
from IPython.display import Markdown, display

summary_path = ANALYSIS_DIR / "summary.md"
summary_text = summary_path.read_text(encoding="utf-8")
display(Markdown(summary_text))


## 10. Archive Outputs


In [ ]:
import zipfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_path = WORK_ROOT / f"v3g_inference_analysis_{timestamp}.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in OUTPUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(WORK_ROOT))

print("Archive:", archive_path)
print("Archive size MB:", round(archive_path.stat().st_size / (1024 * 1024), 2))
